In [1]:
# cell 0: deterministic cuda workspace
import os
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"

In [2]:
# cell 1: imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from scipy.sparse import diags, kron, identity
from scipy.sparse.linalg import spsolve, splu
from scipy.signal import savgol_filter
from scipy.ndimage import uniform_filter1d
from scipy.stats.qmc import LatinHypercube
from scipy import stats
from tqdm import tqdm
import time, math, random, copy, glob, warnings, platform
from itertools import product

warnings.filterwarnings('ignore')
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

In [3]:
# cell 2: seed, device, hardware
seed = 42
torch.manual_seed(seed)
np.random.seed(seed)
random.seed(seed)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

use_cuda = torch.cuda.is_available()
device = torch.device("cuda" if use_cuda else "cpu")
if use_cuda:
    torch.cuda.manual_seed_all(seed)
    torch.cuda.set_device(0)
    torch.set_default_tensor_type('torch.cuda.FloatTensor')

print(f"device: {device}")
print(f"gpu: {torch.cuda.get_device_name(0) if use_cuda else 'none'}")
print(f"cpu: {platform.processor() or platform.machine()}, logical cores: {os.cpu_count()}")
print(f"torch: {torch.__version__}, numpy: {np.__version__}")

device: cpu
gpu: none
cpu: x86_64, logical cores: 4
torch: 2.11.0+cpu, numpy: 2.1.3


In [4]:
# cell 3: frozen noise protocol (unchanged, for reproduction of the original table)
def generate_frozen_noise(n_time_steps, noise_level, seed=seed):
    if noise_level == 0.0:
        return np.zeros(n_time_steps)
    rng = np.random.RandomState(seed)
    return rng.normal(loc=0.0, scale=noise_level, size=n_time_steps)

print("frozen noise protocol defined!")

frozen noise protocol defined!


In [5]:
# cell 4: output directory
output_dir = 'project_outputs'
os.makedirs(output_dir, exist_ok=True)

In [6]:
# cell 5: parameters
alpha = 0.00011644 # thermal diffusivity of copper at 25 c, m^2/s
L = 1.0
Nx = 15
Ny = Nx
Nz = Nx
dx = L / (Nx - 1)
dy = dx
dz = dx
dt = 0.6
t_final = 60
eval_times = np.linspace(1, t_final, 60)

x = np.linspace(0, L, Nx)
y = np.linspace(0, L, Ny)
z = np.linspace(0, L, Nz)
X, Y = np.meshgrid(x, y, indexing='ij')
X_3d, Y_3d, Z_3d = np.meshgrid(x, y, z, indexing='ij')

n_steps = int(round(t_final / dt))

# original indexing (kept for reproduction of the original table)
eval_indices = [min(int(np.round(t / dt)), n_steps - 1) for t in eval_times]
# aligned indexing: snapshot k is the state at time (k+1)*dt
eval_indices_fixed = [int(np.clip(int(np.round(t / dt)) - 1, 0, n_steps - 1)) for t in eval_times]
snap_times_fixed = np.array([(k + 1) * dt for k in eval_indices_fixed])

print(f"grid: {Nx}x{Ny}x{Nz}, dx={dx:.6f}, dt={dt}, n_steps={n_steps}")
print(f"stability: r1d={alpha*dt/dx**2:.5f}, r2d={2*alpha*dt/dx**2:.5f}, r3d={3*alpha*dt/dx**2:.5f}")
print(f"diffusion time L^2/alpha = {L**2/alpha:.0f}s, t_final/tau = {t_final/(L**2/alpha):.4f}, penetration sqrt(alpha*t) = {np.sqrt(alpha*t_final):.4f}")
print(f"original eval_indices[:5]={eval_indices[:5]}, aligned[:5]={eval_indices_fixed[:5]}, snap_times[:5]={snap_times_fixed[:5]}")

grid: 15x15x15, dx=0.071429, dt=0.6, n_steps=100
stability: r1d=0.01369, r2d=0.02739, r3d=0.04108
diffusion time L^2/alpha = 8588s, t_final/tau = 0.0070, penetration sqrt(alpha*t) = 0.0836
original eval_indices[:5]=[2, 3, 5, 7, 8], aligned[:5]=[1, 2, 4, 6, 7], snap_times[:5]=[1.2 1.8 3.  4.2 4.8]


In [7]:
# cell 6: initial and boundary conditions
def tri(v):
    return np.where(v <= L/2, 2 * v / L, 2 * (L - v) / L)

def initial_condition_1d(x, L): return tri(x)
def initial_condition_2d(X, Y, L): return tri(X) * tri(Y)
def initial_condition_3d(X, Y, Z, L): return tri(X) * tri(Y) * tri(Z)

def boundary_conditions_1d(u):
    u[0] = 0.0; u[-1] = 0.0
    return u

def boundary_conditions_2d(u):
    u[0, :] = 0.0; u[-1, :] = 0.0; u[:, 0] = 0.0; u[:, -1] = 0.0
    return u

def boundary_conditions_3d(u):
    u[0, :, :] = 0.0; u[-1, :, :] = 0.0
    u[:, 0, :] = 0.0; u[:, -1, :] = 0.0
    u[:, :, 0] = 0.0; u[:, :, -1] = 0.0
    return u

ui_1d = boundary_conditions_1d(initial_condition_1d(x, L))
ui_2d = boundary_conditions_2d(initial_condition_2d(X, Y, L))
ui_3d = boundary_conditions_3d(initial_condition_3d(X_3d, Y_3d, Z_3d, L))

print(f"ic shapes: {ui_1d.shape}, {ui_2d.shape}, {ui_3d.shape}")
print(f"ic max: {ui_1d.max():.3f}, {ui_2d.max():.3f}, {ui_3d.max():.3f}")

ic shapes: (15,), (15, 15), (15, 15, 15)
ic max: 1.000, 1.000, 1.000


In [8]:
# cell 7: analytical solutions (separable product form), saved as csv
N_analytical = 50

def get_b_coeffs(N):
    n = np.arange(1, N + 1)
    return (8 / (n**2 * np.pi**2)) * np.sin(n * np.pi / 2)

b_coeffs_analytical = get_b_coeffs(N_analytical)

def u1d(xv, t, alpha=alpha, L=L, N=N_analytical, b=b_coeffs_analytical):
    # 1d solution; 2d and 3d are products of this because ic and bcs are separable
    n = np.arange(1, N + 1)
    decay = np.exp(-alpha * (n * np.pi / L) ** 2 * t)
    return (np.sin(np.outer(xv, n) * np.pi / L) * (b * decay)).sum(axis=1)

def ana_fields(times):
    out = {'1d': [], '2d': [], '3d': []}
    for t in times:
        f = u1d(x, t)
        out['1d'].append(f)
        out['2d'].append(np.einsum('i,j->ij', f, f))
        out['3d'].append(np.einsum('i,j,k->ijk', f, f, f))
    return {k: np.stack(v) for k, v in out.items()}

ana_orig = ana_fields(eval_times) # at t = 1..60 (original comparison times)
ana_fixed = ana_fields(snap_times_fixed) # at actual snapshot times (aligned comparison)

# verify against the original 2d triple-loop formula at one time
def analytical_solution_2d_ref(X, Y, t, alpha, L, N, b):
    u = np.zeros_like(X)
    for nx in range(1, N + 1):
        for ny in range(1, N + 1):
            u += b[nx-1] * b[ny-1] * np.sin(nx*np.pi*X/L) * np.sin(ny*np.pi*Y/L) * np.exp(-alpha*((nx*np.pi/L)**2 + (ny*np.pi/L)**2)*t)
    return u

ref = analytical_solution_2d_ref(X, Y, eval_times[30], alpha, L, N_analytical, b_coeffs_analytical)
print(f"max |vectorized - reference| (2d, t={eval_times[30]:.1f}): {np.max(np.abs(ana_orig['2d'][30] - ref)):.2e}")

# save csvs in the original long format (time, x, y, z, temperature)
def to_long(fields, times, dim):
    grids = {'1d': np.meshgrid(x, indexing='ij'),
             '2d': np.meshgrid(x, y, indexing='ij'),
             '3d': np.meshgrid(x, y, z, indexing='ij')}[dim]
    names = ['x', 'y', 'z'][:len(grids)]
    frames = []
    for k, t in enumerate(times):
        d = {'time': np.full(grids[0].size, t)}
        for nm, g in zip(names, grids):
            d[nm] = g.ravel()
        d['temperature'] = fields[k].ravel()
        frames.append(pd.DataFrame(d))
    return pd.concat(frames, ignore_index=True)

analytical_data = {}
for dim in ['1d', '2d', '3d']:
    df = to_long(ana_orig[dim], eval_times, dim)
    analytical_data[dim] = df
    df.to_csv(f'{output_dir}/analytical_solutions_{dim}.csv', index=False)
    print(f"{dim}: {len(df)} rows saved")

print(f"3d analytical range: [{ana_orig['3d'].min():.4f}, {ana_orig['3d'].max():.4f}], t=1 max {ana_orig['3d'][0].max():.4f}, t=60 max {ana_orig['3d'][-1].max():.4f}")

max |vectorized - reference| (2d, t=31.0): 5.55e-16
1d: 900 rows saved
2d: 13500 rows saved
3d: 202500 rows saved
3d analytical range: [0.0000, 0.9285], t=1 max 0.9285, t=60 max 0.5341


In [9]:
# cell 8: fdm engine (backward euler, factorize once, any dim and grid size)
def lap_interior(n, dim, h):
    m = n - 2
    D = (diags([1, -2, 1], [-1, 0, 1], shape=(m, m), format='csc') / h**2).tocsc()
    I = identity(m, format='csc')
    total = None
    for a in range(dim):
        mats = [I] * dim
        mats[a] = D
        term = mats[0]
        for mm in mats[1:]:
            term = kron(term, mm, format='csc')
        total = term if total is None else total + term
    return total.tocsc()

class FDM:
    def __init__(self, dim, n, dt):
        t0 = time.perf_counter()
        dim = int(str(dim)[0])
        self.dim, self.n, self.m = dim, n, n - 2
        self.h = L / (n - 1)
        self.r = alpha * dt / self.h**2
        A = (identity(self.m**dim, format='csc') - alpha * dt * lap_interior(n, dim, self.h)).tocsc()
        self.lu = splu(A)
        nb = np.zeros((self.m,) * dim)
        for a in range(dim):
            for idx in (0, -1):
                sl = [slice(None)] * dim
                sl[a] = idx
                nb[tuple(sl)] += 1
        self.nb = nb.ravel()
        xv = np.linspace(0, L, n)
        self.ic = tri(xv)
        for _ in range(dim - 1):
            self.ic = np.multiply.outer(self.ic, tri(xv))
        self.inner = (slice(1, -1),) * dim
        self.u0_int = self.ic[self.inner].ravel().copy()
        self.setup_s = time.perf_counter() - t0

    def run(self, noise, snap_steps):
        want, out = set(snap_steps), {}
        u = self.u0_int.copy()
        for step in range(len(noise)):
            bc = float(noise[step])
            u = self.lu.solve(u + self.r * bc * self.nb)
            if step in want:
                full = np.full((self.n,) * self.dim, bc)
                full[self.inner] = u.reshape((self.m,) * self.dim)
                out[step] = full
        return np.stack([out[s] for s in snap_steps])

def ana_on(dim, n, times):
    xv = np.linspace(0, L, n)
    F = np.stack([u1d(xv, t) for t in times])
    if dim == '1d': return F
    if dim == '2d': return np.einsum('ti,tj->tij', F, F)
    return np.einsum('ti,tj,tk->tijk', F, F, F)

print("fdm engine defined")

fdm engine defined


In [10]:
# cell 9: metrics (all-node, interior-only, boundary, persistence baseline)
def score(pred, true, ic):
    T = pred.shape[0]
    P, Tt = pred.reshape(T, -1), true.reshape(T, -1)
    n, dim = pred.shape[1], pred.ndim - 1
    mask = np.zeros((n,) * dim, bool)
    mask[(slice(1, -1),) * dim] = True
    m = mask.ravel()
    U0 = np.broadcast_to(ic.ravel()[None], Tt.shape)
    nrm = np.linalg.norm
    d = P - Tt
    return {
        'rel_all': nrm(d) / nrm(Tt),
        'rel_int': nrm(d[:, m]) / nrm(Tt[:, m]),
        'bnd_rms': float(np.sqrt(np.mean(d[:, ~m] ** 2))),
        'bnd_err_share': float(np.sum(d[:, ~m] ** 2) / np.sum(d ** 2)),
        'bnd_node_frac': float(1 - m.mean()),
        'persist_all': nrm(U0 - Tt) / nrm(Tt),
        'persist_int': nrm((U0 - Tt)[:, m]) / nrm(Tt[:, m]),
    }

print("metrics defined")

metrics defined


In [11]:
# cell 10: part 1 - reproduce original table, then interior/boundary split
ORIG_ACC = {  # original implicit-fdm accuracy %, from your table
    ('1d', 0.00): 99.211881, ('1d', 0.05): 97.050712, ('1d', 0.10): 94.261812, ('1d', 0.15): 91.438071, ('1d', 0.20): 88.605361,
    ('2d', 0.00): 98.731993, ('2d', 0.05): 92.669227, ('2d', 0.10): 85.500436, ('2d', 0.15): 78.295180, ('2d', 0.20): 71.080701,
    ('3d', 0.00): 98.276686, ('3d', 0.05): 84.014035, ('3d', 0.10): 68.160206, ('3d', 0.15): 52.275435, ('3d', 0.20): 36.382896}
PINN_REL = {  # old pinn all-node rel l2, from logs
    ('1d', 0.00): 0.002762, ('1d', 0.05): 0.006487, ('1d', 0.10): 0.011117, ('1d', 0.15): 0.015973, ('1d', 0.20): 0.019401,
    ('2d', 0.00): 0.008447, ('2d', 0.05): 0.013196, ('2d', 0.10): 0.022862, ('2d', 0.15): 0.033210, ('2d', 0.20): 0.045272,
    ('3d', 0.00): 0.015825, ('3d', 0.05): 0.056281, ('3d', 0.10): 0.057742, ('3d', 0.15): 0.068396, ('3d', 0.20): 0.092494}

noise_levels = [0.0, 0.05, 0.10, 0.15, 0.20]
fd15 = {d: FDM(d, 15, dt) for d in ['1d', '2d', '3d']}
rows = []
for dim in ['1d', '2d', '3d']:
    f = fd15[dim]
    for s in noise_levels:
        nv = generate_frozen_noise(n_steps, s, seed=seed)
        # original alignment (reproduction)
        so = f.run(nv, eval_indices)
        ro = score(so, ana_orig[dim], f.ic)
        acc_o = max(0, (1 - ro['rel_all']) * 100)
        # aligned snapshots vs analytical at true snapshot times
        sa = f.run(nv, eval_indices_fixed)
        ra = score(sa, ana_fixed[dim], f.ic)
        rows.append({'dim': dim, 'sigma': s, 'acc_orig': acc_o, 'acc_table': ORIG_ACC[(dim, s)],
                     'check': 'PASS' if abs(acc_o - ORIG_ACC[(dim, s)]) < 0.05 else 'FAIL',
                     **{k: ra[k] for k in ['rel_all', 'rel_int', 'bnd_rms', 'bnd_err_share', 'bnd_node_frac', 'persist_all', 'persist_int']},
                     'pinn_rel_all_old': PINN_REL[(dim, s)]})

df_part1 = pd.DataFrame(rows)
df_part1.to_csv(f'{output_dir}/part1_split.csv', index=False)
print("reproduction check (original alignment):")
print(df_part1[['dim', 'sigma', 'acc_orig', 'acc_table', 'check']].to_string(index=False, float_format='%.4f'))
print("\ninterior/boundary split (aligned snapshots):")
print(df_part1[['dim', 'sigma', 'rel_all', 'rel_int', 'bnd_rms', 'bnd_err_share', 'bnd_node_frac',
                'persist_all', 'persist_int', 'pinn_rel_all_old']].to_string(index=False, float_format='%.5f'))

reproduction check (original alignment):
dim  sigma  acc_orig  acc_table check
 1d 0.0000   99.2119    99.2119  PASS
 1d 0.0500   97.0507    97.0507  PASS
 1d 0.1000   94.2618    94.2618  PASS
 1d 0.1500   91.4381    91.4381  PASS
 1d 0.2000   88.6054    88.6054  PASS
 2d 0.0000   98.7320    98.7320  PASS
 2d 0.0500   92.6692    92.6692  PASS
 2d 0.1000   85.5004    85.5004  PASS
 2d 0.1500   78.2952    78.2952  PASS
 2d 0.2000   71.0807    71.0807  PASS
 3d 0.0000   98.2767    98.2767  PASS
 3d 0.0500   84.0140    84.0140  PASS
 3d 0.1000   68.1602    68.1602  PASS
 3d 0.1500   52.2754    52.2754  PASS
 3d 0.2000   36.3829    36.3829  PASS

interior/boundary split (aligned snapshots):
dim   sigma  rel_all  rel_int  bnd_rms  bnd_err_share  bnd_node_frac  persist_all  persist_int  pinn_rel_all_old
 1d 0.00000  0.00887  0.00887  0.00000        0.00000        0.13333      0.07197      0.07197           0.00276
 1d 0.05000  0.03319  0.00913  0.04693        0.92436        0.13333      0.071

In [12]:
# cell 11: part 2 - fdm refinement ladder with full timing (r2.2)
ladder = {'1d': [15, 31, 61, 121, 241], '2d': [15, 31, 61, 121], '3d': [15, 23, 31]}
PINN_TIME = {'1d': 68.5, '2d': 221.9, '3d': 894.2}
rows = []
for dim, ns in ladder.items():
    for n in ns:
        t0 = time.perf_counter()
        f = FDM(dim, n, dt)
        snaps = eval_indices_fixed
        truth = ana_on(dim, n, snap_times_fixed)
        for s in [0.0, 0.20]:
            nv = generate_frozen_noise(n_steps, s, seed=seed)
            t1 = time.perf_counter()
            sol = f.run(nv, snaps)
            run_s = time.perf_counter() - t1
            r = score(sol, truth, f.ic)
            rows.append({'dim': dim, 'n': n, 'unknowns': (n - 2) ** int(dim[0]), 'sigma': s,
                         'setup_s': f.setup_s, 'run_s': run_s, 'total_s': f.setup_s + run_s,
                         'rel_all': r['rel_all'], 'rel_int': r['rel_int'], 'bnd_rms': r['bnd_rms'],
                         'pinn_time_s': PINN_TIME[dim], 'pinn_rel_all_clean': PINN_REL[(dim, 0.0)],
                         'pinn_rel_all_s02': PINN_REL[(dim, 0.2)]})
        print(f"{dim} n={n} done, setup {f.setup_s:.2f}s")

df_ladder = pd.DataFrame(rows)
df_ladder.to_csv(f'{output_dir}/part2_ladder.csv', index=False)
print(df_ladder.to_string(index=False, float_format='%.5g'))

1d n=15 done, setup 0.00s
1d n=31 done, setup 0.00s
1d n=61 done, setup 0.00s
1d n=121 done, setup 0.00s
1d n=241 done, setup 0.00s
2d n=15 done, setup 0.00s
2d n=31 done, setup 0.01s
2d n=61 done, setup 0.03s
2d n=121 done, setup 0.11s
3d n=15 done, setup 0.05s
3d n=23 done, setup 0.69s
3d n=31 done, setup 5.07s
dim   n  unknowns  sigma   setup_s     run_s   total_s    rel_all    rel_int    bnd_rms  pinn_time_s  pinn_rel_all_clean  pinn_rel_all_s02
 1d  15        13      0  0.001562 0.0022297 0.0037917  0.0088749  0.0088749 5.1832e-17         68.5            0.002762          0.019401
 1d  15        13    0.2  0.001562  0.002194  0.003756    0.12822   0.012252    0.18772         68.5            0.002762          0.019401
 1d  31        29      0 0.0015379 0.0022589 0.0037968  0.0020846  0.0020846 5.1832e-17         68.5            0.002762          0.019401
 1d  31        29    0.2 0.0015379 0.0022744 0.0038123   0.088326   0.014087    0.18772         68.5            0.002762         

In [13]:
# cell 12: part 3 - denoised-fdm baselines over 30 noise seeds
seeds = list(range(1000, 1030))
tt = np.arange(n_steps) / n_steps

def denoise(v, method):
    if method == 'none': return v
    if method == 'moving_avg5': return uniform_filter1d(v, size=5, mode='nearest')
    if method == 'savgol11_2': return savgol_filter(v, 11, 2)
    if method == 'poly3': return np.polyval(np.polyfit(tt, v, 3), tt)
    if method == 'oracle_zero': return np.zeros_like(v)

methods = ['none', 'moving_avg5', 'savgol11_2', 'poly3', 'oracle_zero']
rows = []
for dim in ['1d', '2d', '3d']:
    f = fd15[dim]
    for s in [0.10, 0.20]:
        for sd in seeds:
            raw = generate_frozen_noise(n_steps, s, seed=sd)
            for mth in methods:
                nv = denoise(raw, mth)
                sol = f.run(nv, eval_indices_fixed)
                r = score(sol, ana_fixed[dim], f.ic)
                rows.append({'dim': dim, 'sigma': s, 'seed': sd, 'method': mth,
                             'resid_noise_std': float(np.std(nv)), 'rel_all': r['rel_all'],
                             'rel_int': r['rel_int'], 'bnd_rms': r['bnd_rms']})

df_den = pd.DataFrame(rows)
df_den.to_csv(f'{output_dir}/part3_denoised.csv', index=False)
summ = df_den.groupby(['dim', 'sigma', 'method'])[['resid_noise_std', 'rel_all', 'rel_int', 'bnd_rms']].agg(['mean', 'std'])
summ.columns = ['_'.join(c) for c in summ.columns]
summ = summ.reset_index()
summ['pinn_rel_all_old'] = [PINN_REL[(d, s)] for d, s in zip(summ['dim'], summ['sigma'])]
print(summ.to_string(index=False, float_format='%.5f'))

dim   sigma      method  resid_noise_std_mean  resid_noise_std_std  rel_all_mean  rel_all_std  rel_int_mean  rel_int_std  bnd_rms_mean  bnd_rms_std  pinn_rel_all_old
 1d 0.10000 moving_avg5               0.04324              0.00563       0.03200      0.00407       0.00979      0.00080       0.04478      0.00608           0.01112
 1d 0.10000        none               0.09894              0.00566       0.06859      0.00447       0.00982      0.00078       0.09983      0.00658           0.01112
 1d 0.10000 oracle_zero               0.00000              0.00000       0.00887      0.00000       0.00887      0.00000       0.00000      0.00000           0.01112
 1d 0.10000       poly3               0.01461              0.00613       0.01598      0.00410       0.00966      0.00076       0.01835      0.00700           0.01112
 1d 0.10000  savgol11_2               0.04416              0.00629       0.03263      0.00428       0.00980      0.00078       0.04574      0.00644           0.01112
 1d 

In [14]:
# cell 13: part 3b - low-order bc fits (mean, linear, quadratic)
def denoise2(v, m):
    if m == 'mean': return np.full_like(v, v.mean())
    if m == 'poly1': return np.polyval(np.polyfit(tt, v, 1), tt)
    if m == 'poly2': return np.polyval(np.polyfit(tt, v, 2), tt)

rows = []
for dim in ['1d', '2d', '3d']:
    f = fd15[dim]
    for s in [0.10, 0.20]:
        for sd in seeds:
            raw = generate_frozen_noise(n_steps, s, seed=sd)
            for mth in ['mean', 'poly1', 'poly2']:
                nv = denoise2(raw, mth)
                r = score(f.run(nv, eval_indices_fixed), ana_fixed[dim], f.ic)
                rows.append({'dim': dim, 'sigma': s, 'seed': sd, 'method': mth,
                             'resid_noise_std': float(np.std(nv)), 'rel_all': r['rel_all'],
                             'rel_int': r['rel_int'], 'bnd_rms': r['bnd_rms']})
df_den2 = pd.DataFrame(rows)
df_den2.to_csv(f'{output_dir}/part3b_lowfit.csv', index=False)
s2 = df_den2.groupby(['dim', 'sigma', 'method'])[['resid_noise_std', 'rel_all', 'rel_int']].agg(['mean', 'std'])
s2.columns = ['_'.join(c) for c in s2.columns]
s2 = s2.reset_index()
s2['pinn_rel_all_old'] = [PINN_REL[(d, s)] for d, s in zip(s2['dim'], s2['sigma'])]
print(s2.to_string(index=False, float_format='%.5f'))

dim   sigma method  resid_noise_std_mean  resid_noise_std_std  rel_all_mean  rel_all_std  rel_int_mean  rel_int_std  pinn_rel_all_old
 1d 0.10000   mean               0.00000              0.00000       0.01192      0.00308       0.00943      0.00061           0.01112
 1d 0.10000  poly1               0.00688              0.00519       0.01329      0.00343       0.00958      0.00067           0.01112
 1d 0.10000  poly2               0.01039              0.00520       0.01425      0.00381       0.00962      0.00073           0.01112
 1d 0.20000   mean               0.00000              0.00000       0.01751      0.00804       0.01081      0.00201           0.01940
 1d 0.20000  poly1               0.01377              0.01038       0.02112      0.00850       0.01132      0.00219           0.01940
 1d 0.20000  poly2               0.02077              0.01041       0.02353      0.00902       0.01143      0.00234           0.01940
 2d 0.10000   mean               0.00000              0.00000 

In [15]:
# cell 14: part 5 - time-varying true boundary (fdm only, cpu)
A_BC = 0.5
SIG = [0.10, 0.20]
SEEDS5 = list(range(2000, 2030))
REF = {'1d': (225, 16), '2d': (113, 8), '3d': (43, 3)} # (n_ref, subsample factor to the 15-node grid)
Q = 4 # fine/coarse time-step ratio for the reference
HORIZONS = {'short_60s': dt, 'long_0.35tau': 30.0} # dt per horizon (n_steps = 100 in both)
tt5 = np.arange(1, n_steps + 1) / n_steps
methods5 = ['none', 'mean', 'moving_avg5', 'savgol11_2', 'poly3']

def g_true(tsec, kind, T):
    if kind == 'ramp':
        return A_BC * tsec / T
    return A_BC * np.sin(2 * np.pi * tsec / (T / 2))

def den5(v, m):
    if m == 'none': return v
    if m == 'mean': return np.full_like(v, v.mean())
    if m == 'moving_avg5': return uniform_filter1d(v, 5, mode='nearest')
    if m == 'savgol11_2': return savgol_filter(v, 11, 2)
    if m == 'poly3': return np.polyval(np.polyfit(tt5, v, 3), tt5)

rows = []
for hname, dt_h in HORIZONS.items():
    T = n_steps * dt_h
    t_coarse = np.arange(1, n_steps + 1) * dt_h
    t_fine = np.arange(1, Q * n_steps + 1) * (dt_h / Q)
    fine_steps = [(k + 1) * Q - 1 for k in eval_indices_fixed]
    for dim in ['1d', '2d', '3d']:
        d = int(dim[0])
        n_ref, fct = REF[dim]
        t0 = time.perf_counter()
        fref = FDM(dim, n_ref, dt_h / Q)
        f = FDM(dim, 15, dt_h)
        print(f"{hname} {dim}: reference setup {time.perf_counter()-t0:.1f}s")
        for kind in ['ramp', 'sine']:
            case = f"{hname}|{kind}"
            ref = fref.run(g_true(t_fine, kind, T), fine_steps)
            truth = ref[(slice(None),) + (slice(None, None, fct),) * d]
            assert truth.shape[1:] == (15,) * d, truth.shape
            g_c = g_true(t_coarse, kind, T)

            # oracle: true g(t), no noise
            r = score(f.run(g_c, eval_indices_fixed), truth, f.ic)
            rows.append({'case': case, 'dim': dim, 'sigma': 0.0, 'seed': -1, 'method': 'oracle_true_g',
                         'rel_all': r['rel_all'], 'rel_int': r['rel_int'], 'bnd_rms': r['bnd_rms'],
                         'bc_rmse': 0.0, 'persist_all': r['persist_all']})
            for s in SIG:
                for sd in SEEDS5:
                    obs = g_c + generate_frozen_noise(n_steps, s, seed=sd)
                    for m in methods5:
                        nv = den5(obs, m)
                        r = score(f.run(nv, eval_indices_fixed), truth, f.ic)
                        rows.append({'case': case, 'dim': dim, 'sigma': s, 'seed': sd, 'method': m,
                                     'rel_all': r['rel_all'], 'rel_int': r['rel_int'], 'bnd_rms': r['bnd_rms'],
                                     'bc_rmse': float(np.sqrt(np.mean((nv - g_c) ** 2))),
                                     'persist_all': r['persist_all']})
            print(f"  {case} {dim} done")
        del fref

df5 = pd.DataFrame(rows)
df5.to_csv(f'{output_dir}/part5_timevarying.csv', index=False)

full = df5.groupby(['case', 'dim', 'sigma', 'method'])[['rel_all', 'rel_int', 'bc_rmse']].agg(['mean', 'std'])
full.columns = ['_'.join(c) for c in full.columns]
full.reset_index().to_csv(f'{output_dir}/part5_summary.csv', index=False)

order = ['oracle_true_g', 'none', 'mean', 'moving_avg5', 'savgol11_2', 'poly3']
sub = df5[df5['sigma'].isin([0.0, 0.2])]
for met in ['rel_all', 'rel_int', 'bc_rmse']:
    p = sub.groupby(['case', 'dim', 'method'])[met].mean().unstack('method')[order]
    print(f"\n{met} (mean over 30 seeds, sigma=0.2; oracle has no noise):")
    print(p.to_string(float_format='%.5f'))
print("\npersistence baseline (rel_all):")
print(df5.groupby(['case', 'dim'])['persist_all'].mean().to_string(float_format='%.5f'))
print("\nstd over seeds, rel_all, sigma=0.2:")
print(sub[sub.method != 'oracle_true_g'].groupby(['case', 'dim', 'method'])['rel_all'].std().unstack('method')[order[1:]].to_string(float_format='%.5f'))

short_60s 1d: reference setup 0.0s
  short_60s|ramp 1d done
  short_60s|sine 1d done
short_60s 2d: reference setup 0.1s
  short_60s|ramp 2d done
  short_60s|sine 2d done
short_60s 3d: reference setup 49.9s
  short_60s|ramp 3d done
  short_60s|sine 3d done
long_0.35tau 1d: reference setup 0.0s
  long_0.35tau|ramp 1d done
  long_0.35tau|sine 1d done
long_0.35tau 2d: reference setup 0.1s
  long_0.35tau|ramp 2d done
  long_0.35tau|sine 2d done
long_0.35tau 3d: reference setup 50.2s
  long_0.35tau|ramp 3d done
  long_0.35tau|sine 3d done

rel_all (mean over 30 seeds, sigma=0.2; oracle has no noise):
method                 oracle_true_g    none    mean  moving_avg5  savgol11_2   poly3
case              dim                                                                
long_0.35tau|ramp 1d         0.01207 0.25214 0.30039      0.13515     0.13774 0.07501
                  2d         0.01026 0.40389 0.44603      0.21303     0.21730 0.11184
                  3d         0.00657 0.47035 0.47869  